# African High School Journal Platform - UI-Integrated Colab Notebook

This notebook reconstructs and tests version **0.2.2** of the current GitHub-ready Streamlit project. It includes the repaired workflow and feature-card grids, the responsive fixed-header safe area, the full Python journal workflow, migrated UI assets, and all regression tests.

**Founder / Author / Main Builder:** Kavya Kaushal Shah  
**Advisor / Mentor:** Dr. Qingyang Xiao


## 1. Recreate the deployable repository

This cell extracts the current UI-integrated Streamlit project from an embedded compressed archive. The notebook file itself is intentionally excluded from its own embedded archive to avoid recursive packaging.


In [ ]:
from pathlib import Path
import base64
import io
import shutil
import zipfile

PROJECT_NAME = "african_high_school_journal_platform"
ARCHIVE_B64 = """..."""

project_dir = Path(PROJECT_NAME)
if project_dir.exists():
    shutil.rmtree(project_dir)

with zipfile.ZipFile(io.BytesIO(base64.b64decode(ARCHIVE_B64))) as zf:
    zf.extractall(Path.cwd())

print("Recreated:", project_dir.resolve())
print("Files:", sum(1 for path in project_dir.rglob("*") if path.is_file()))


## 2. Install dependencies

Streamlit and scikit-learn may take a minute to install in a fresh Colab runtime.


In [ ]:
%cd african_high_school_journal_platform
!python --version
!python -m pip install -q -r requirements.txt


## 3. Run service and UI tests

The service smoke test validates submission, similarity screening, reviewer matching, review, and publication. The UI tests validate assets and all six app workspaces. The HTML-grid regression test confirms that all workflow and feature cards render as HTML instead of appearing as visible markup. The top-header regression test verifies that Streamlit's fixed toolbar cannot cover the journal masthead on desktop or mobile.


In [ ]:
!python tests/smoke_test.py
!python tests/ui_integration_test.py
!python tests/app_static_execution_test.py
!python tests/html_grid_regression_test.py
!python tests/top_header_layout_regression_test.py


## 4. Launch Streamlit

For local Jupyter, uncomment the direct Streamlit command. In Colab, the optional ngrok block can create a temporary public demo URL. Streamlit Community Cloud remains the recommended stable deployment.


In [ ]:
# Local launch:
# !streamlit run app.py

RUN_TUNNEL = False
if RUN_TUNNEL:
    import getpass
    import subprocess
    import time
    from pathlib import Path

    !python -m pip install -q pyngrok
    from pyngrok import ngrok

    token = getpass.getpass("Paste ngrok authtoken: ")
    if token:
        ngrok.set_auth_token(token)
    log_path = Path("streamlit.log")
    process = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
        stdout=log_path.open("w"),
        stderr=subprocess.STDOUT,
    )
    time.sleep(8)
    public_url = ngrok.connect(8501)
    print("Temporary Streamlit URL:", public_url)
    print("Process ID:", process.pid)
else:
    print("Tunnel not started. Set RUN_TUNNEL = True only for a temporary demo.")


## 5. Create a GitHub-ready ZIP

Run this cell after making notebook edits. The ZIP can be downloaded from the Colab file browser.


In [ ]:
from pathlib import Path
import zipfile

project_root = Path.cwd()
zip_path = project_root.parent / "african_high_school_journal_platform_top_layout_fix.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in project_root.rglob("*"):
        if not path.is_file():
            continue
        if "__pycache__" in path.parts or ".pytest_cache" in path.parts:
            continue
        if path.suffix in {".pyc", ".db", ".sqlite"}:
            continue
        zf.write(path, arcname=Path(project_root.name) / path.relative_to(project_root))
print("Created:", zip_path.resolve())


## Deployment reminder

Upload the contents of the project folder to the GitHub repository root and use `app.py` as the Streamlit entrypoint. Version 0.2.2 preserves the earlier compact card-grid repair and adds responsive clearance beneath Streamlit Community Cloud's fixed top toolbar, keeping the date strip, masthead, credits, and navigation fully visible.
